In [27]:
!pip install opendatasets

In [28]:
import opendatasets as od
od.download("https://www.kaggle.com/datasets/henryshan/google-stock-price")

Skipping, found downloaded files in "./google-stock-price" (use force=True to force download)


In [29]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

df = pd.read_csv("/content/google-stock-price/GOOG.csv")
data = df[['Open','High','Low','Close','Volume']].values

close_prices = df['Close'].values.reshape(-1,1)

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(close_prices)

sequence_length = 60
X = []
y = []

for i in range(sequence_length, len(scaled_data)-1):
    X.append(scaled_data[i-sequence_length:i, 0])

    # Direction: 1 if tomorrow price > today price
    if scaled_data[i+1, 0] > scaled_data[i, 0]:
        y.append(1)
    else:
        y.append(0)

X = np.array(X)
y = np.array(y)

X = np.reshape(X, (X.shape[0], X.shape[1], 1))

In [30]:
df

,Date,Open,High,Low,Close,Adj Close,Volume
0,2004-08-19,2.490664,2.591785,2.390042,2.499133,2.499133,897427216
1,2004-08-20,2.515820,2.716817,2.503118,2.697639,2.697639,458857488
2,2004-08-23,2.758411,2.826406,2.716070,2.724787,2.724787,366857939
3,2004-08-24,2.770615,2.779581,2.579581,2.611960,2.611960,306396159
4,2004-08-25,2.614201,2.689918,2.587302,2.640104,2.640104,184645512
...,...,...,...,...,...,...,...
4853,2023-11-29,138.985001,139.669998,136.294998,136.399994,136.399994,21014700
4854,2023-11-30,136.399994,136.960007,132.789993,133.919998,133.919998,29913500
4855,2023-12-01,133.320007,133.500000,132.151993,133.320007,133.320007,24258400
4856,2023-12-04,131.294006,131.449997,129.399994,130.630005,130.630005,24117100


In [31]:
df.head()

,Date,Open,High,Low,Close,Adj Close,Volume
0,2004-08-19,2.490664,2.591785,2.390042,2.499133,2.499133,897427216
1,2004-08-20,2.515820,2.716817,2.503118,2.697639,2.697639,458857488
2,2004-08-23,2.758411,2.826406,2.716070,2.724787,2.724787,366857939
3,2004-08-24,2.770615,2.779581,2.579581,2.611960,2.611960,306396159
4,2004-08-25,2.614201,2.689918,2.587302,2.640104,2.640104,184645512


In [32]:
df.shape

(4858, 7)

In [33]:
scaler = MinMaxScaler(feature_range=(0,1))
scaled_data = scaler.fit_transform(data)

In [34]:
split = int(0.8 * len(X))

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

In [35]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential()

model.add(LSTM(50, return_sequences=True, input_shape=(X_train.shape[1],1)))
model.add(Dropout(0.2))

model.add(LSTM(50))
model.add(Dropout(0.2))

model.add(Dense(1, activation='sigmoid'))   # ← Changed

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',   # ← Changed
    metrics=['accuracy']          # ← Now works!
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [36]:
model.fit(X_train, y_train, epochs=20, batch_size=32)

Epoch 1/20
120/120 ━━━━━━━━━━━━━━━━━━━━ 6s 25ms/step - accuracy: 0.5209 - loss: 0.6930
Epoch 2/20
120/120 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.5072 - loss: 0.6932
Epoch 3/20
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.5194 - loss: 0.6929
Epoch 4/20
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.4960 - loss: 0.6939
Epoch 5/20
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.5145 - loss: 0.6930
Epoch 6/20
120/120 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.5220 - loss: 0.6921
Epoch 7/20
120/120 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.5172 - loss: 0.6928
Epoch 8/20
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.5094 - loss: 0.6935
Epoch 9/20
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.5203 - loss: 0.6924
Epoch 10/20
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.5215 - loss: 0.6923
Epoch 11/20
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.5205 - loss: 0.6927
Epoch 12/20
120/120 ━━━━━━━━━━━━━━━━━━━━ 

In [37]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", accuracy)

30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5501 - loss: 0.6899
Test Accuracy: 0.534375011920929
